In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras

model_path = (
    "/content/drive/MyDrive/"
    "dogs_cats_classification/cats_dogs_model.keras"
)

model = tf_keras.models.load_model(
    model_path,
    custom_objects={"KerasLayer": hub.KerasLayer},
    compile=False
)



model.summary()

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 1280)              2257984   
                                                                 
 dense (Dense)               (None, 2)                 2562      
                                                                 
Total params: 2260546 (8.62 MB)
Trainable params: 2562 (10.01 KB)
Non-trainable params: 2257984 (8.61 MB)
_________________________________________________________________


In [3]:
import numpy as np
from PIL import Image

image_path = "/content/dog.4001.jpg"  # استبدله بمسار صورة الاختبار

img = Image.open(image_path).convert("RGB")
img = img.resize((224, 224))

x = np.array(img, dtype=np.float32) / 255.0
x = np.expand_dims(x, axis=0)

predictions = model.predict(x)
predicted_class = np.argmax(predictions[0])

print("قيم النموذج:", predictions[0])
print("رقم الفئة المتوقعة:", predicted_class)

1/1 [==============================] - 3s 3s/step
قيم النموذج: [-1.4240922  0.2173242]
رقم الفئة المتوقعة: 1


In [4]:
!pip install -q gradio

In [5]:
def predict_image(image):
    if image is None:
        return None, {"يرجى رفع صورة": 1.0}

    # تحويل الصورة إلى RGB وتغيير حجمها
    image = image.convert("RGB").resize((224, 224))

    # تجهيز الصورة بالطريقة نفسها المستخدمة في التدريب
    image_array = np.array(image, dtype=np.float32) / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    # التنبؤ
    logits = model.predict(image_array, verbose=0)

    # تحويل القيم الخام إلى احتمالات
    probabilities = tf.nn.softmax(logits[0]).numpy()

    # يجب أن يطابق ترتيب الأسماء ترتيب الترميز أثناء التدريب
    class_names = ["Cat", "Dog"]

    results = {
        class_names[i]: float(probabilities[i])
        for i in range(len(class_names))
    }

    predicted_class = class_names[int(np.argmax(probabilities))]
    confidence = float(np.max(probabilities)) * 100

    return (
        image,
        {
            predicted_class: float(np.max(probabilities)),
            **{
                name: prob
                for name, prob in results.items()
                if name != predicted_class
            }
        }
    )

In [7]:
import gradio as gr
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="ارفع صورة قط أو كلب"),
    outputs=[
        gr.Image(label="الصورة المدخلة"),
        gr.Label(num_top_classes=2, label="الصنف والاحتمالية")
    ],
    title="تصنيف صور القطط والكلاب",
    description=(
        "ارفع صورة للحصول على الصنف المتوقع "
        "واحتمالية كل من القط والكلب."
    ),
    examples=None
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dee2882908dfea04f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
